In [1]:
import pandas as pd
import numpy as np

industry = pd.read_csv("../data/industry_dataset.csv")
syllabus = pd.read_csv("../data/syllabus_dataset.csv")

print('Industry shape:', industry.shape)
print('Syllabus shape:', syllabus.shape)
industry.head()

Industry shape: (712, 5)
Syllabus shape: (331, 4)


,Company,Year,Role,Skill,Skill_Type
0,Oracle,2021,Backend Developer,Java,Traditional
1,Oracle,2021,Backend Developer,Spring Boot,Transitional
2,Oracle,2021,Backend Developer,SQL,Transitional
3,Oracle,2021,Software Engineer,Java,Traditional
4,Oracle,2021,Software Engineer,Data Structures,Transitional


In [ ]:
industry['Skill'] = industry['Skill'].str.lower().str.strip()
syllabus['Skill']  = syllabus['Skill'].str.lower().str.strip()

industry['Year'] = pd.to_numeric(industry['Year'])

industry['Upcoming_Flag'] = (
    industry['Skill_Type'].str.lower() == 'upcoming'
).astype(int)

print('Basic cleaning done')
industry.head()

Basic cleaning done


,Company,Year,Role,Skill,Skill_Type,Upcoming_Flag
0,Oracle,2021,Backend Developer,java,Traditional,0
1,Oracle,2021,Backend Developer,spring boot,Transitional,0
2,Oracle,2021,Backend Developer,sql,Transitional,0
3,Oracle,2021,Software Engineer,java,Traditional,0
4,Oracle,2021,Software Engineer,data structures,Transitional,0


In [ ]:
skill_normalization = {
    'reactjs':              'react',
    'nodejs':               'node.js',
    'python basics':        'python',
    'html/css':             'html',
    'statistical analysis': 'statistics',
    'multi-cloud strategy': 'multi-cloud',
    'devops practices':     'devops',
    'search algorithms':    'searching algorithms',
    'vulnerability assessment': 'vulnerability scanning',
    'query optimization':   'sql optimization',
    'neural networks':      'deep learning',
}

syllabus['Skill'] = syllabus['Skill'].replace(skill_normalization)

print('Normalization applied')
print('Syllabus unique skills after normalization:', syllabus['Skill'].nunique())

Normalization applied
Syllabus unique skills after normalization: 97


In [4]:
ind_skills = set(industry['Skill'].unique())
syl_skills = set(syllabus['Skill'].unique())

overlap       = ind_skills & syl_skills
true_gaps     = ind_skills - syl_skills
only_syllabus = syl_skills - ind_skills

print(f'Industry skills  : {len(ind_skills)}')
print(f'Syllabus skills  : {len(syl_skills)}')
print(f'Overlap          : {len(overlap)}')
print(f'True gaps        : {len(true_gaps)}')
print(f'Only in syllabus : {len(only_syllabus)}')
print()
print('Skills covered by curriculum:')
print(sorted(overlap))

Industry skills  : 95
Syllabus skills  : 97
Overlap          : 28
True gaps        : 67
Only in syllabus : 69

Skills covered by curriculum:
['advanced sql', 'aws', 'azure', 'cloud architecture', 'cloud security', 'data visualization', 'deep learning', 'distributed systems', 'feature engineering', 'html', 'java', 'javascript', 'kubernetes', 'machine learning', 'microservices', 'mlops', 'model deployment', 'multi-cloud', 'node.js', 'python', 'react', 'rest apis', 'sql', 'sql optimization', 'statistics', 'system design', 'vulnerability scanning', 'zero trust architecture']


In [ ]:
TOTAL_INSTITUTES = syllabus['Institute'].nunique()
print(f'Total institutes in syllabus: {TOTAL_INSTITUTES}')
print('Institutes:', syllabus['Institute'].unique())

taught_counts = (
    syllabus.groupby('Skill')['Institute']
    .nunique()
    .reset_index()
    .rename(columns={'Institute': 'Taught_Institute_Count'})
)

msrit_skills = set(
    syllabus[syllabus['Institute'] == 'MSRIT']['Skill'].unique()
)
print(f'MSRIT teaches {len(msrit_skills)} skills')

industry = industry.merge(taught_counts, on='Skill', how='left')
industry['Taught_Institute_Count'] = industry['Taught_Institute_Count'].fillna(0).astype(int)

industry['Is_Taught_MSRIT'] = industry['Skill'].apply(
    lambda s: 1 if s in msrit_skills else 0
)

industry['Institutional_Gap_Score'] = (
    (TOTAL_INSTITUTES - industry['Taught_Institute_Count']) / TOTAL_INSTITUTES
).round(3)

print('Columns added: Taught_Institute_Count, Is_Taught_MSRIT, Institutional_Gap_Score')
print(industry[['Skill','Is_Taught_MSRIT','Taught_Institute_Count','Institutional_Gap_Score']].drop_duplicates('Skill').head(15))

Total institutes in syllabus: 5
Institutes: ['RVCE' 'MSRIT' 'BIT' 'PESIT' 'DSCE']
MSRIT teaches 69 skills
Columns added: Taught_Institute_Count, Is_Taught_MSRIT, Institutional_Gap_Score
               Skill  Is_Taught_MSRIT  Taught_Institute_Count  \
0               java                1                       4   
1        spring boot                0                       0   
2                sql                1                       5   
4    data structures                0                       0   
6             python                1                       4   
7         statistics                1                       3   
8   machine learning                1                       5   
9               html                1                       4   
10               css                0                       0   
11        javascript                1                       4   
12               aws                1                       5   
13        networking              

In [ ]:
dup_industry = industry.duplicated().sum()
dup_syllabus = syllabus.duplicated().sum()
print(f'Industry duplicates: {dup_industry}')
print(f'Syllabus duplicates: {dup_syllabus}')

industry = industry.drop_duplicates()
syllabus  = syllabus.drop_duplicates()

print('Final industry shape:', industry.shape)
print('Final syllabus shape:', syllabus.shape)

Industry duplicates: 2
Syllabus duplicates: 3
Final industry shape: (710, 9)
Final syllabus shape: (328, 4)


In [7]:
print('=== FINAL CLEANED INDUSTRY DATASET ===')
print('Shape:', industry.shape)
print('Columns:', industry.columns.tolist())
print('Nulls:')
print(industry.isnull().sum())
print()
print('Skill_Type distribution:')
print(industry['Skill_Type'].value_counts())
print()
print('MSRIT gap summary:')
msrit_gaps = industry[industry['Is_Taught_MSRIT']==0]['Skill'].nunique()
total_skills = industry['Skill'].nunique()
print(f'  Total unique industry skills : {total_skills}')
print(f'  Missing from MSRIT           : {msrit_gaps}')
print(f'  Gap percentage               : {round(msrit_gaps/total_skills*100,1)}%')
industry.head()

=== FINAL CLEANED INDUSTRY DATASET ===
Shape: (710, 9)
Columns: ['Company', 'Year', 'Role', 'Skill', 'Skill_Type', 'Upcoming_Flag', 'Taught_Institute_Count', 'Is_Taught_MSRIT', 'Institutional_Gap_Score']
Nulls:
Company                    0
Year                       0
Role                       0
Skill                      0
Skill_Type                 0
Upcoming_Flag              0
Taught_Institute_Count     0
Is_Taught_MSRIT            0
Institutional_Gap_Score    0
dtype: int64

Skill_Type distribution:
Skill_Type
Transitional    385
Traditional     212
Upcoming        113
Name: count, dtype: int64

MSRIT gap summary:
  Total unique industry skills : 95
  Missing from MSRIT           : 73
  Gap percentage               : 76.8%


,Company,Year,Role,Skill,Skill_Type,Upcoming_Flag,Taught_Institute_Count,Is_Taught_MSRIT,Institutional_Gap_Score
0,Oracle,2021,Backend Developer,java,Traditional,0,4,1,0.2
1,Oracle,2021,Backend Developer,spring boot,Transitional,0,0,0,1.0
2,Oracle,2021,Backend Developer,sql,Transitional,0,5,1,0.0
3,Oracle,2021,Software Engineer,java,Traditional,0,4,1,0.2
4,Oracle,2021,Software Engineer,data structures,Transitional,0,0,0,1.0


In [8]:
industry.to_csv('../outputs/cleaned_industry.csv', index=False)
syllabus.to_csv('../outputs/cleaned_syllabus.csv', index=False)

print('Saved:')
print('  ../outputs/cleaned_industry.csv')
print('  ../outputs/cleaned_syllabus.csv')

Saved:
  ../outputs/cleaned_industry.csv
  ../outputs/cleaned_syllabus.csv
